In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 29. テキスト テキスト テキスト テキスト

> **学習目標**
> - テキスト テキスト(TP)テキスト 行列テキスト テキスト テキスト テキスト
> - テキスト テキスト(PP)テキスト テキスト(bubble) 問題テキスト テキスト
> - 1F1B テキスト テキスト

## 29.1 テキスト モデル テキスト テキスト

データ テキスト: モデルテキスト テキスト GPUテキスト テキスト テキスト. 70B+ モデルテキスト テキスト テキスト.

テキスト:
- **テキスト テキスト (TP)**: テキスト テキスト テキスト テキスト GPUテキスト テキスト
- **テキスト テキスト (PP)**: テキスト テキスト GPUテキスト テキスト
- **3D テキスト**: DP + TP + PP テキスト (GPT-3 学習 テキスト)

## 29.2 テキスト テキスト (Megatron-LM テキスト)

テキスト $Y = XW$テキスト テキスト:

### Column Parallel (テキスト テキスト)
$W = [W_1; W_2]$ (テキスト テキスト)
$$Y = XW = X[W_1; W_2] = [XW_1, XW_2]$$
- テキスト GPUテキスト $W_i$ 計算, 結果 $XW_i$ テキスト
- 出力テキスト テキスト All-Gather

### Row Parallel (テキスト テキスト)
$W = [W_1, W_2]$ (テキスト テキスト)
$$Y = XW = [X_1, X_2][W_1; W_2] = X_1 W_1 + X_2 W_2$$
- テキスト GPUテキスト $X_i W_i$ 計算
- All-Reduceテキスト テキスト

Megatron: Attentionテキスト column, FFNテキスト column + row テキスト All-Reduce 1テキスト.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Column Parallel Linear テキスト
class ColumnParallelLinear(nn.Module):
    """Wテキスト テキスト テキスト (テキスト テキスト)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert out_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.out_per_gpu = out_features // n_gpus
        # テキスト GPUテキスト Weight
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, self.out_per_gpu) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # テキスト GPUテキスト テキスト Subspace Computation
        outputs = [x @ w for w in self.weights]
        # All-Gather: Resultテキスト テキスト
        return torch.cat(outputs, dim=-1)

# Row Parallel Linear
class RowParallelLinear(nn.Module):
    """Wテキスト テキスト テキスト (Input度 テキスト)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert in_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.in_per_gpu = in_features // n_gpus
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(self.in_per_gpu, out_features) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # Input度 テキスト
        x_chunks = torch.chunk(x, self.n_gpus, dim=-1)
        # テキスト GPU Computation
        partial = [xc @ w for xc, w in zip(x_chunks, self.weights)]
        # All-Reduce: Sumテキスト
        return sum(partial)

# Test
d_in, d_out = 64, 64
n_gpus = 2

# Original
torch.manual_seed(0)
W_full = torch.randn(d_in, d_out) * 0.1

# Column parallel テキスト
col = ColumnParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    # Weightテキスト W_fullテキスト テキスト Setting
    for i in range(n_gpus):
        col.weights[i].copy_(W_full[:, i*col.out_per_gpu:(i+1)*col.out_per_gpu])

x = torch.randn(4, d_in)
y_full = x @ W_full
y_col = col(x)
print(f"Column Parallel 誤差: {(y_full - y_col).abs().max():.2e}")

# Row parallel
row = RowParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    for i in range(n_gpus):
        row.weights[i].copy_(W_full[i*row.in_per_gpu:(i+1)*row.in_per_gpu, :])
y_row = row(x)
print(f"Row Parallel Error: {(y_full - y_row).abs().max():.2e}")
print("\n=> テキスト テキスト テキスト SumテキストPlane Originalテキスト テキスト!")


## 29.3 Attentionテキスト テキスト テキスト

MHAテキスト テキスト TPテキスト テキスト: テキスト テキスト テキスト GPUテキスト テキスト.

- $h$テキスト テキスト, $n$ GPU → テキスト GPUテキスト $h/n$テキスト テキスト
- テキスト $W_i^Q, W_i^K, W_i^V$ 計算 (テキスト)
- テキスト 結果 Concat → All-Reduceテキスト テキスト

## 29.4 テキスト テキスト

テキスト テキスト GPUテキスト テキスト:
- GPU 0: テキスト 1-4
- GPU 1: テキスト 5-8
- GPU 2: テキスト 9-12

順伝播テキスト GPU 0 → 1 → 2 テキスト. バックプロパゲーションテキスト テキスト.

### テキスト 問題
テキスト GPUテキスト テキスト テキスト テキスト GPUテキスト テキスト → テキスト 時間(テキスト) テキスト.

テキスト テキスト: $\frac{p-1}{m + p - 1}$
- $p$: テキスト テキスト テキスト
- $m$: テキスト テキスト

テキスト: **テキスト** — テキスト $m$テキスト テキスト テキスト テキスト.


In [ ]:
# テキスト テキスト 可視化
def pipeline_bubble_ratio(p, m):
    """Bubble Ratio."""
    return (p - 1) / (m + p - 1)

# テキスト p, mテキスト テキスト
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bubble Ratio
ax = axes[0]
for p in [2, 4, 8, 16]:
    ms = np.arange(1, 50)
    ratios = [pipeline_bubble_ratio(p, m) for m in ms]
    ax.plot(ms, ratios, label=f'p={p}', linewidth=2)
ax.set_xlabel('テキストBatch テキスト m')
ax.set_ylabel('Bubble Ratio')
ax.set_title('Pipeline Bubble Ratio')
ax.legend(); ax.grid(True, alpha=0.3)

# Gantt テキスト (テキスト)
ax = axes[1]
p, m = 4, 8
# テキスト GPUテキスト テキスト テキスト (Forward Pass F, Backward Pass B, テキスト .)
schedule = []
for stage in range(p):
    s = []
    # forward: stage テキスト
    for micro in range(m):
        # テキスト Time = stage + micro
        for t in range(stage + micro + 1):
            pass
    # テキスト forward テキスト backward
    s = ['.'] * (2 * m + 2 * (p - 1))
    # forward Pattern
    for micro in range(m):
        start = stage + micro
        if start < len(s):
            s[start] = f'F{micro}'
    # backward
    for micro in range(m):
        start = m + 2 * (p - 1) - stage + micro
        if 0 <= start < len(s):
            s[start] = f'B{micro}'
    schedule.append(s)

# Visualization (テキスト)
for i, s in enumerate(schedule):
    for j, x in enumerate(s):
        if x.startswith('F'):
            ax.barh(i, 1, left=j, color='blue', alpha=0.7)
        elif x.startswith('B'):
            ax.barh(i, 1, left=j, color='red', alpha=0.7)
        else:
            ax.barh(i, 1, left=j, color='gray', alpha=0.3)
ax.set_xlabel('Time Step')
ax.set_ylabel('GPU (stage)')
ax.set_title(f'Pipeline テキスト (p={p}, m={m})')
ax.set_yticks(range(p))
plt.tight_layout()
plt.savefig('../figures/ch29_pipeline_bubble.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"p=4, m=8 テキスト テキスト: {pipeline_bubble_ratio(4, 8)*100:.1f}%")
print(f"p=4, m=32 Bubble Ratio: {pipeline_bubble_ratio(4, 32)*100:.1f}%")
print("=> テキストBatch テキスト テキスト Bubble Decrease.")


## 29.5 1F1B テキスト

テキスト テキスト: テキスト forward テキスト テキスト backward → テキスト テキスト.

**1F1B**: forward 1テキスト, backward 1テキスト テキスト. テキスト値 テキスト テキスト.


## 29.6 3D テキスト (DP + TP + PP)

GPT-3 (175B) 学習 テキスト:
- DP: データ テキスト (テキスト テキスト)
- TP: Attention/FFN テキスト テキスト
- PP: テキスト テキスト

テキスト: 1024 GPU = 8 DP × 8 TP × 16 PP

## 29.7 要点

| テキスト | テキスト テキスト | テキスト | テキスト |
|---|---|---|---|
| データ (DP) | データ | All-Reduce (テキスト) | モデル テキスト |
| テキスト (TP) | テキスト 行列 | All-Reduce / All-Gather | テキスト テキスト |
| テキスト (PP) | テキスト | Point-to-Point | テキスト テキスト + テキスト値 |
| 3D | テキスト | テキスト | テキスト |

## 演習問題

1. Column parallelテキスト row parallelテキスト テキスト テキスト テキスト 結果テキスト テキスト テキスト.
2. テキスト テキスト テキスト p=2, 4, 8テキスト m=4, 16, 64テキスト テキスト 計算テキスト.
3. Attention テキスト 4 GPUテキスト テキスト テキスト テキスト テキスト.
4. 3D テキスト 1024 GPUテキスト DP=8, TP=8, PP=16テキスト 内容テキスト テキスト テキスト.
5. 1F1B テキスト テキスト テキスト テキスト テキスト.

> 解答: `solutions/ch29_solutions.ipynb`
